In [ ]:
from feature_utils import *


In [1]:
import os
import h5py
import torch
import librosa
import pandas as pd
import numpy as np
import torch.nn.functional as F
from transformers import Wav2Vec2FeatureExtractor, HubertModel

# ====== Configuration ======
AUDIO_DIR = "../data/raw_data/alexander_nygaard19/sound_stimuli"
EXCEL_PATH = "../data/raw_data/alexander_nygaard19/AN19-exposure-test-behavioral-data.xlsx"

# Define configurations for both base and fine-tuned models
MODEL_CONFIGS = [
    {
        "model_id": "facebook/hubert-large-ll60k",
        "feat_output": "../data/features/nygaard19_features.h5",
        "tsne_output": "../data/features/nygaard19_tsne_3d.h5"
    },
    {
        "model_id": "facebook/hubert-large-ls960-ft",
        "feat_output": "../data/features/nygaard19_features_ft.h5",
        "tsne_output": "../data/features/nygaard19_tsne_3d_ft.h5"
    }
]

layers_spec = {
    "cnn": [2, 3, 4, 5, 6],
    "tr":  [0, 2, 4, 6, 8, 10, 12, 14, 16, 18, 20, 22, 24], 
}


In [2]:
# ========================




# ====== Execution Logic ======
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Executing on device: {device}")

import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='openpyxl')

# Read behavioral data and extract unique filenames (Learning + Test phases)
print("Loading behavioral data to filter required audio files...")
df = pd.read_excel(EXCEL_PATH)
used_filenames = df['FileName'].dropna().unique()
used_filenames_lower = set(f.lower() for f in used_filenames)
print(f"Found {len(used_filenames_lower)} unique audio files used in the experiment.")

# Retrieve valid .wav or .WAV files matching the Excel file recursively
audio_paths = []
for root, dirs, files in os.walk(AUDIO_DIR):
    for file in files:
        if file.lower().endswith('.wav'):
            audio_paths.append(os.path.join(root, file))

print(f"Successfully located {len(audio_paths)} matching audio files in directory.")

if not audio_paths:
    print("Error: Matching audio files not found.")
else:
    for config in MODEL_CONFIGS:
        model_id = config["model_id"]
        output_h5 = config["feat_output"]
        
        print(f"\n--- Initializing Model: {model_id} ---")
        processor = Wav2Vec2FeatureExtractor.from_pretrained(model_id)
        model = HubertModel.from_pretrained(model_id).to(device)
        model.eval()

        with h5py.File(output_h5, "w") as h5f:
            for each_path in audio_paths:
                file_basename = os.path.basename(each_path)
                
                # Filename parsing logic based on 'kf1ew01 was.wav' format
                speaker_id = file_basename[:3].lower()
                word_name = file_basename.lower().split(" ")[-1][:-4]
                
                if speaker_id not in h5f:
                    speaker_group = h5f.create_group(speaker_id)
                else:
                    speaker_group = h5f[speaker_id]
                
                audio, sr = librosa.load(each_path, sr=None, mono=True)
                wave_res = librosa.resample(audio, orig_sr=sr, target_sr=16000)
                
                inputs = processor(wave_res, sampling_rate=16000, return_tensors="pt", padding=False)
                input_values = inputs.input_values.to(device)
                
                layer_feats = _extract_selected_layers(model, input_values, layers_spec)
                
                if word_name in speaker_group:
                    del speaker_group[word_name]
                word_group = speaker_group.create_group(word_name)
                
                for layer_key, feat_np in layer_feats.items():
                    word_group.create_dataset(layer_key, data=feat_np, compression="gzip")
                    
            print(f"Successfully processed all files for model {model_id}!")


Executing on device: cuda
Loading behavioral data to filter required audio files...
Found 1056 unique audio files used in the experiment.
Successfully located 6261 matching audio files in directory.

--- Initializing Model: facebook/hubert-large-ll60k ---


c:\Users\Alex\anaconda3\envs\BayesPCN\lib\site-packages\transformers\models\hubert\modeling_hubert.py:762: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:555.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


Successfully processed all files for model facebook/hubert-large-ll60k!

--- Initializing Model: facebook/hubert-large-ls960-ft ---
Successfully processed all files for model facebook/hubert-large-ls960-ft!


In [3]:
import h5py
import numpy as np
from joblib import Parallel, delayed
from sklearn.manifold import TSNE
import multiprocessing
import time
import pickle
import os

TSNE_DIM = 3

# We will load the old pickle file to get the exact ordering of (speaker, word)
old_pkl_path = r'C:\Users\Alex\Desktop\Pycharm\cross-talker-ASR\July\hubert_Nygaard_all_tr_dict.pkl'
if os.path.exists(old_pkl_path):
    with open(old_pkl_path, 'rb') as f:
        old_data = pickle.load(f)
    # Get order from a layer, e.g., layer 24
    old_spk_word_order = []
    # In old data, keys are integer layers, e.g. 24
    if 24 in old_data:
        for spk, words in old_data[24].items():
            for word in words.keys():
                old_spk_word_order.append((spk, word))
        print(f"Extracted {len(old_spk_word_order)} ordered elements from old pkl.")
    else:
        print("Warning: layer 24 not found in old pkl.")
        old_spk_word_order = None
else:
    print("Warning: Old pkl not found, will fallback to alphabetical order.")
    old_spk_word_order = None



def run_tsne_on_features():
    for config in MODEL_CONFIGS:
        feat_h5 = config["feat_output"]
        tsne_h5 = config["tsne_output"].replace(".h5", "../data/features/_random.h5")
        
        if not os.path.exists(feat_h5):
            print(f"File {feat_h5} not found. Skip.")
            continue
            
        print(f"\nProcessing t-SNE for {feat_h5} -> {tsne_h5}")
        layers = get_all_layers(feat_h5)
        print(f"Found {len(layers)} layers: {layers}")
        
        start_time = time.time()
        num_cores = max(1, multiprocessing.cpu_count() - 2)
        print(f"Running t-SNE in parallel on {num_cores} cores...")
        
        results = Parallel(n_jobs=num_cores)(
            delayed(process_single_layer)(layer, feat_h5) for layer in layers
        )
        
        print(f"Saving results to {tsne_h5}...")
        with h5py.File(tsne_h5, "w") as f_out:
            for layer_name, layer_dict in results:
                for spk, words in layer_dict.items():
                    for word, matrix in words.items():
                        group_path = f"{spk}/{word}/{layer_name}"
                        f_out.create_dataset(group_path, data=matrix, compression="gzip")
                        
        elapsed = time.time() - start_time
        print(f"Finished {tsne_h5} in {elapsed:.2f} seconds.")

# if __name__ == "__main__":
#     run_tsne_on_features()


Extracted 6231 ordered elements from old pkl.


In [4]:
run_tsne_on_features()


Processing t-SNE for nygaard19_features.h5 -> nygaard19_tsne_3d_random.h5
Found 18 layers: ['tr_10', 'cnn_6', 'tr_18', 'tr_22', 'tr_6', 'tr_14', 'cnn_2', 'tr_24', 'tr_16', 'tr_20', 'tr_8', 'tr_4', 'tr_2', 'cnn_5', 'cnn_4', 'tr_0', 'tr_12', 'cnn_3']
Running t-SNE in parallel on 30 cores...
Saving results to nygaard19_tsne_3d_random.h5...
Finished nygaard19_tsne_3d_random.h5 in 21518.01 seconds.

Processing t-SNE for nygaard19_features_ft.h5 -> nygaard19_tsne_3d_ft_random.h5
Found 18 layers: ['tr_10', 'cnn_6', 'tr_18', 'tr_22', 'tr_6', 'tr_14', 'cnn_2', 'tr_24', 'tr_16', 'tr_20', 'tr_8', 'tr_4', 'tr_2', 'cnn_5', 'cnn_4', 'tr_0', 'tr_12', 'cnn_3']
Running t-SNE in parallel on 30 cores...
Saving results to nygaard19_tsne_3d_ft_random.h5...
Finished nygaard19_tsne_3d_ft_random.h5 in 16267.42 seconds.
